In [ ]:
import sqlite3
from pathlib import Path

SQLITE_PATH = Path(
    r"C:\PROYECTO\DoctorPlagio\data\copypaste\copypaste_index.sqlite3"
)

if not SQLITE_PATH.exists():
    raise FileNotFoundError(f"No existe: {SQLITE_PATH}")

conn = sqlite3.connect(SQLITE_PATH)

tables = conn.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""").fetchall()

print("Tablas encontradas:")
for table in tables:
    print(" -", table[0])

In [ ]:
for (table_name,) in tables:
    print("\n" + "=" * 80)
    print(f"TABLA: {table_name}")
    print("=" * 80)

    columns = conn.execute(
        f'PRAGMA table_info("{table_name}")'
    ).fetchall()

    for column in columns:
        print(column)

for (table_name,) in tables:
    count = conn.execute(
        f'SELECT COUNT(*) FROM "{table_name}"'
    ).fetchone()[0]

    print(f"{table_name}: {count:,} registros")        

In [ ]:
import sqlite3
from pathlib import Path

SQLITE_PATH = Path(
    r"C:\PROYECTO\DoctorPlagio\data\copypaste\copypaste_index.sqlite3"
)

conn = sqlite3.connect(SQLITE_PATH)

print("=" * 80)
print("ESTRUCTURA DE fingerprints")
print("=" * 80)

columns = conn.execute("""
    PRAGMA table_info(fingerprints)
""").fetchall()

for col in columns:
    print(
        f"posición={col[0]} | "
        f"nombre={col[1]} | "
        f"tipo={col[2]} | "
        f"not_null={col[3]} | "
        f"default={col[4]} | "
        f"pk={col[5]}"
    )

print("\n" + "=" * 80)
print("ÍNDICES DE fingerprints")
print("=" * 80)

indexes = conn.execute("""
    SELECT name, sql
    FROM sqlite_master
    WHERE type = 'index'
      AND tbl_name = 'fingerprints'
    ORDER BY name
""").fetchall()

for name, sql in indexes:
    print(name)
    print(sql)

conn.close()

In [ ]:
import sqlite3
import psycopg2
from pathlib import Path
from datetime import datetime
import time
import os
import gc

SQLITE_PATH = Path(
    r"C:\PROYECTO\DoctorPlagio\data\copypaste\copypaste_index.sqlite3"
)

PG_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "drplagium",
    "user": "postgres",
    "password": "123456789$",
}

BATCH_DOCUMENTS = 50_000
BATCH_FINGERPRINTS = 100_000

print("SQLite:", SQLITE_PATH)
print("Existe:", SQLITE_PATH.exists())

In [ ]:
sqlite_conn = sqlite3.connect(
    str(SQLITE_PATH),
    timeout=60
)

pg_conn = psycopg2.connect(**PG_CONFIG)

print("✓ SQLite conectado")
print("✓ PostgreSQL conectado")

sqlite_cur = sqlite_conn.cursor()
pg_cur = pg_conn.cursor()

In [ ]:
pg_cur.execute("""
DROP TABLE IF EXISTS corpus_fingerprints;
DROP TABLE IF EXISTS corpus_documents;
""")

pg_conn.commit()

pg_cur.execute("""
CREATE TABLE corpus_documents (
    id BIGINT PRIMARY KEY,
    source TEXT NOT NULL,
    relative_path TEXT NOT NULL,
    title TEXT,
    language TEXT,
    license TEXT,
    source_url TEXT,
    source_file TEXT,
    source_locator TEXT,
    content_hash TEXT NOT NULL,
    token_count INTEGER NOT NULL DEFAULT 0,
    created_at TIMESTAMP NOT NULL
);
""")

pg_cur.execute("""
CREATE TABLE corpus_fingerprints (
    fingerprint TEXT NOT NULL,
    document_id BIGINT NOT NULL,
    position INTEGER NOT NULL,
    PRIMARY KEY (fingerprint, document_id, position)
);
""")

pg_conn.commit()

print("✓ Tablas creadas")

In [ ]:
import io

def copy_documents_batch(rows):
    buffer = io.StringIO()

    for row in rows:
        values = []

        for value in row:
            if value is None:
                values.append(r"\N")
            else:
                text = str(value)
                text = text.replace("\\", "\\\\")
                text = text.replace("\t", "\\t")
                text = text.replace("\n", "\\n")
                text = text.replace("\r", "\\r")
                values.append(text)

        buffer.write("\t".join(values) + "\n")

    buffer.seek(0)

    pg_cur.copy_from(
        buffer,
        "corpus_documents",
        columns=(
            "id",
            "source",
            "relative_path",
            "title",
            "language",
            "license",
            "source_url",
            "source_file",
            "source_locator",
            "content_hash",
            "token_count",
            "created_at",
        ),
        null="\\N",
    )

In [ ]:
start = time.time()

sqlite_cur.execute("""
SELECT
    id,
    source,
    relative_path,
    title,
    language,
    license,
    source_url,
    source_file,
    source_locator,
    content_hash,
    token_count,
    created_at
FROM corpus_documents
ORDER BY id
""")

total_documents = sqlite_cur.execute(
    "SELECT COUNT(*) FROM corpus_documents"
).fetchone()[0]

processed = 0

while True:

    rows = sqlite_cur.fetchmany(BATCH_DOCUMENTS)

    if not rows:
        break

    copy_documents_batch(rows)

    pg_conn.commit()

    processed += len(rows)

    elapsed = time.time() - start
    rate = processed / elapsed if elapsed else 0

    print(
        f"Documentos: {processed:,}/{total_documents:,} "
        f"({processed / total_documents * 100:.2f}%) "
        f"| {rate:,.0f} docs/s"
    )

print("\n✓ corpus_documents terminado")
print(f"Tiempo: {(time.time() - start)/60:.2f} minutos")

In [ ]:
from pathlib import Path

EXPORT_DIR = Path(r"C:\PROYECTO\DoctorPlagio\data\copypaste_export")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Exportando a:", EXPORT_DIR)

In [ ]:
import sqlite3
import csv
import time
from pathlib import Path

SQLITE_PATH = Path(
    r"C:\PROYECTO\DoctorPlagio\data\copypaste\copypaste_index.sqlite3"
)

conn = sqlite3.connect(str(SQLITE_PATH))
conn.execute("PRAGMA query_only = ON")

output = EXPORT_DIR / "corpus_documents.tsv"

start = time.time()

with open(output, "w", encoding="utf-8", newline="") as f:

    writer = csv.writer(
        f,
        delimiter="\t",
        quoting=csv.QUOTE_MINIMAL,
        lineterminator="\n"
    )

    writer.writerow([
        "id",
        "source",
        "relative_path",
        "title",
        "language",
        "license",
        "source_url",
        "source_file",
        "source_locator",
        "content_hash",
        "token_count",
        "created_at",
    ])

    cursor = conn.execute("""
        SELECT
            id,
            source,
            relative_path,
            title,
            language,
            license,
            source_url,
            source_file,
            source_locator,
            content_hash,
            token_count,
            created_at
        FROM corpus_documents
        ORDER BY id
    """)

    count = 0

    while True:

        rows = cursor.fetchmany(50_000)

        if not rows:
            break

        writer.writerows(rows)
        count += len(rows)

        print(
            f"\rDocumentos exportados: {count:,}",
            end=""
        )

print()
print("✓ Exportación de documentos terminada")
print("Archivo:", output)
print("Registros:", f"{count:,}")
print("Tamaño:", f"{output.stat().st_size / (1024**3):.2f} GB")
print("Tiempo:", f"{(time.time() - start)/60:.2f} minutos")

In [ ]:
print("=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)

for file in EXPORT_DIR.iterdir():

    if file.is_file():

        print(
            f"{file.name:30} "
            f"{file.stat().st_size / (1024**3):8.2f} GB"
        )

In [ ]:
import chromadb
from pathlib import Path

CHROMA_PATH = Path(
    r"C:\PROYECTO\DoctorPlagio\chatbox\db\chroma_data"
)

client = chromadb.PersistentClient(
    path=str(CHROMA_PATH)
)

print("Colecciones:")
for c in client.list_collections():
    print(" -", c.name)

In [28]:
collection = client.get_collection(
    "tesis_universitarias_v11"
)

print("Colección:", collection.name)
print("Cantidad:", collection.count())

Colección: tesis_universitarias_v11
Cantidad: 1356


In [29]:
data = collection.get(
    limit=3,
    include=[
        "embeddings",
        "documents",
        "metadatas"
    ]
)

print("IDs:")
print(data["ids"])

print("\nDocumentos:")
for doc in data["documents"]:
    print(repr(doc[:200] if doc else None))

print("\nMetadatos:")
print(data["metadatas"])

print("\nDimensiones de embeddings:")

for embedding in data["embeddings"]:
    print(len(embedding))

IDs:
['FTG7  GRUPO #10 - SISTEMA DE CONTROL DE GESTIÓN Y OPTIMIZACION DE LA ENTREGA DE RESPUESTOS EN FÁBRICAS DISTRIBUCIÓN MASIVA (CASO\xa0FRITO\xa0LAY).pdf_0', 'FTG7  GRUPO #10 - SISTEMA DE CONTROL DE GESTIÓN Y OPTIMIZACION DE LA ENTREGA DE RESPUESTOS EN FÁBRICAS DISTRIBUCIÓN MASIVA (CASO\xa0FRITO\xa0LAY).pdf_1', 'FTG7  GRUPO #10 - SISTEMA DE CONTROL DE GESTIÓN Y OPTIMIZACION DE LA ENTREGA DE RESPUESTOS EN FÁBRICAS DISTRIBUCIÓN MASIVA (CASO\xa0FRITO\xa0LAY).pdf_2']

Documentos:
' \n  \n \nUNIVERSIDAD DOMINICANA O & M \nFUNDADA EL 12 DE ENERO DE 1966 \n \nDIRECCION DE TRABAJO FINAL DE GRADO Y CURSO \nMONOGRÁFICO \n \nINFORME FINAL DE MONOGRÁFICO PARA OPTAR POR EL TITULO DE \nINGENIERO E'
' \n \n \n \n \n \nTEMA \n \nSISTEMA DE CONTROL DE GESTIÓN Y OPTIMIZACION DE LA \nENTREGA DE RESPUESTOS EN FÁBRICAS DISTRIBUCIÓN \nMASIVA (CASO: FRITO LAY) \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \ni \n \n \n \n \n \n \n \n \n \n '
'o las dificultades parecían \nmayores. Este logro también es t